# Text-Based Adventure Game: Space Station Escape

**Lumexa Python Foundations · Course 2: Problem Solving & Program Design · Lessons 2–3**

An interactive game with decisions, branching scenarios, reusable functions and structured data.

* **Part 1 · Lesson 2:** small functions that each do one job.
* **Part 2 · Lesson 3:** nested dictionaries for the rooms and the player.

▶ Run the code cell below (**Shift + Enter**) and type your answers when the program asks. To keep your own copy, choose **File → Save a copy in Drive**.

In [ ]:
# ======================================================================
#  TEXT-BASED ADVENTURE GAME: SPACE STATION ESCAPE
#  Lumexa Python Foundations · Course 2: Problem Solving & Program Design
#  Part 1 · Lesson 2: functions in depth and modular programming
#  Part 2 · Lesson 3: nested collections (a world of rooms, a player record)
# ======================================================================


# ----------------------------------------------------------------------
# PART 1 · Lesson 2: small functions that each do one job
# ----------------------------------------------------------------------

def show_title(title, symbol="=", width=44):
    """Print a banner. `symbol` and `width` have default values."""
    line = symbol * width
    print(line)
    print(title)
    print(line)


def get_choice(prompt, valid_choices):
    """Keep asking until the player types one of the valid choices, then return it."""
    while True:
        text = input(prompt).strip().lower()
        if text in valid_choices:
            return text
        print("Please choose one of:", ", ".join(valid_choices))


def split_command(text):
    """Return TWO values: the verb and the noun, so 'Take  Keycard' gives ('take', 'keycard')."""
    words = text.strip().lower().split()
    if len(words) == 0:
        return "", ""
    verb = words[0]
    noun = " ".join(words[1:])
    return verb, noun


def show_room(name, description, exits, items):
    """Show one room. Call it with keyword arguments to make the call easy to read."""
    print()
    print("📍 " + name)
    print(description)
    print("Exits:", ", ".join(exits))
    if len(items) > 0:
        print("You see:", ", ".join(items))


def breathe(oxygen, amount=1):
    """A parameter is a LOCAL variable: the function changes its own copy and returns the result."""
    return oxygen - amount


# A quick test of the tools, before the game uses them.
verb, noun = split_command("  Take   KEYCARD ")           # unpack the two returned values
print("Command test:", verb, "|", noun)
show_room(name="Test Room", description="Only for testing.", exits=["north"], items=["coin"])
print("Oxygen after one breath:", breathe(10), "| after a deep breath:", breathe(10, amount=3))


# ----------------------------------------------------------------------
# PART 2 · Lesson 3: the world is a nested dictionary
# ----------------------------------------------------------------------
# Every room is a dictionary that holds a description, a dictionary of exits and a list of items.
world = {
    "Corridor": {
        "description": "A long corridor. Doors lead west to the lab, east to the garden, north to the engine room and south to the airlock.",
        "exits": {"west": "Lab", "east": "Garden", "north": "Engine Room", "south": "Airlock"},
        "items": [],
    },
    "Lab": {
        "description": "A lab full of dusty monitors. A keycard lies on a desk.",
        "exits": {"east": "Corridor"},
        "items": ["keycard"],
    },
    "Garden": {
        "description": "A quiet greenhouse. A spare oxygen tank hangs on the wall.",
        "exits": {"west": "Corridor"},
        "items": ["oxygen tank"],
    },
    "Engine Room": {
        "description": "The engines hum. A heavy door leads north to the escape pod. It needs a keycard.",
        "exits": {"south": "Corridor", "north": "Escape Pod"},
        "items": ["wrench"],
    },
    "Airlock": {
        "description": "A cold airlock. Through the window you can see the stars. Nothing useful here.",
        "exits": {"north": "Corridor"},
        "items": [],
    },
    "Escape Pod": {
        "description": "The escape pod is fuelled and ready to launch!",
        "exits": {"south": "Engine Room"},
        "items": [],
    },
}

# The player is a record: a dictionary with the location, an inventory list and the oxygen left.
player = {"location": "Corridor", "inventory": [], "oxygen": 7}


def show_current_room():
    room = world[player["location"]]
    show_room(name=player["location"], description=room["description"], exits=sorted(room["exits"]), items=room["items"])
    print("Oxygen:", player["oxygen"], "moves left")


def move_player(direction):
    """Move through an exit. Returns False when the move is not possible."""
    room = world[player["location"]]
    if direction not in room["exits"]:
        print("You can't go that way.")
        return False
    destination = room["exits"][direction]
    if destination == "Escape Pod" and "keycard" not in player["inventory"]:
        print("The door is locked. You need a keycard.")
        return False
    player["location"] = destination
    player["oxygen"] = breathe(player["oxygen"])         # every move costs one breath
    return True


def take(item):
    room = world[player["location"]]
    if item in room["items"]:
        room["items"].remove(item)
        player["inventory"].append(item)
        print("You take the " + item + ".")
    else:
        print("There is no " + item + " here.")


def use(item):
    if item not in player["inventory"]:
        print("You don't have a " + item + ".")
    elif item == "oxygen tank":
        player["inventory"].remove(item)
        player["oxygen"] += 5
        print("You connect the tank. Oxygen:", player["oxygen"], "moves left.")
    else:
        print("You can't use the " + item + " right now.")


def show_inventory():
    if len(player["inventory"]) == 0:
        print("You are carrying nothing.")
    else:
        names = [item.title() for item in player["inventory"]]          # a list comprehension
        print("You are carrying:", ", ".join(names))


def play():
    show_title("SPACE STATION ESCAPE")
    print("Your oxygen is running out. Find the keycard and reach the escape pod!")
    print("Commands: go <direction>, take <item>, use <item>, inventory, look, help, quit")
    show_current_room()

    while True:
        text = input("\n> What do you do? ")
        verb, noun = split_command(text)

        if verb == "go":
            if move_player(noun):
                if player["location"] == "Escape Pod":
                    print("\n🚀 You launch the escape pod. YOU ESCAPED!")
                    break
                show_current_room()
        elif verb == "take":
            take(noun)
        elif verb == "use":
            use(noun)
        elif verb in ("inventory", "i"):
            show_inventory()
        elif verb == "look":
            show_current_room()
        elif verb == "help":
            print("Commands: go <direction>, take <item>, use <item>, inventory, look, help, quit")
        elif verb == "quit":
            print("You abandon the mission. Goodbye!")
            break
        elif verb == "":
            print("Type a command, or help.")
        else:
            print("I don't understand '" + text.strip() + "'. Type help.")

        if player["oxygen"] <= 0:
            print("\n💨 Your oxygen has run out. GAME OVER.")
            break


def main():
    ready = get_choice("Ready to play? (yes/no) ", ["yes", "no"])
    if ready == "yes":
        play()
    else:
        print("Maybe next time!")


# This guard means the game starts when the file is run, but not when it is imported.
if __name__ == "__main__":
    main()